# CI/CD, Continuous Training & MLOps Maturity

Companion notebook for the [CI/CD & Continuous Training lesson](https://ml-viz-ruby.vercel.app/courses/ml-in-practice/16-cicd-and-continuous-training).

**The idea in one sentence.** Models **decay** as the world drifts away from their training
data, so mature MLOps automates **continuous training** (retrain on a schedule or when drift
is detected) behind a **promotion gate** that only ships a candidate if it beats production
and clears a quality floor.

The core ideas:

- **Model decay:** accuracy falls over time without retraining.
- **Scheduled vs drift-triggered CT:** retrain every N days, or only when accuracy dips
  below a threshold — drift-triggered often matches scheduled with fewer retrains.
- **Promotion gate:** automation must never ship a regression — promote only a validated
  improvement.

We simulate all three and **validate the decay and the promotion gate**, then cover the
gotchas.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Dark style matching the site theme.
plt.style.use('dark_background')
plt.rcParams.update({
    'axes.edgecolor': '#475569',
    'axes.labelcolor': '#e2e8f0',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
    'axes.titlecolor': '#e2e8f0',
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d27',
    'grid.color': '#2e3347',
    'savefig.facecolor': '#0f1117',
})
BRAND = '#6366f1'
TEAL = '#14b8a6'
ROSE = '#f43f5e'
YELLOW = '#eab308'

rng = np.random.default_rng(0)

## 1. A model decays under drift

The data distribution shifts over time. A model frozen at day 0 sees its accuracy fall as the world moves away from its training distribution. We model accuracy decaying with the drift between 'now' and the last training time.

In [ ]:
T = 120  # days
drift_rate = 0.004
def accuracy(days_since_train):
    # starts at 0.92, decays with drift, floored
    return 0.5 + 0.42 * np.exp(-drift_rate * days_since_train)

days = np.arange(T)
frozen = accuracy(days)  # never retrained
print('day 0 acc:', round(frozen[0],3), ' day 119 acc:', round(frozen[-1],3))

### Validate: a frozen model decays under drift

Without retraining, accuracy falls as the data drifts. We confirm the frozen model's
accuracy is monotonically decreasing — the whole motivation for continuous training.

In [ ]:
print(f'day 0 accuracy: {frozen[0]:.3f}, day {T-1} accuracy: {frozen[-1]:.3f}')
assert frozen[-1] < frozen[0], 'a frozen model decays as the world drifts'
assert all(frozen[i] >= frozen[i+1] - 1e-9 for i in range(len(frozen)-1)), 'decay is monotonic here'
print('\n✅ models decay without retraining — this is why continuous training exists')

## 2. Continuous Training: scheduled vs drift-triggered

- **Scheduled CT** retrains every `period` days (accuracy resets toward fresh).
- **Drift-triggered CT** retrains only when accuracy drops below a threshold `theta`.

We simulate both and compare average accuracy and number of retrains.

In [ ]:
def simulate_ct(policy, period=20, theta=0.80):
    acc = np.zeros(T); last_train = 0; retrains = 0
    for d in range(T):
        a = accuracy(d - last_train)
        if policy == 'scheduled' and d > 0 and d % period == 0:
            last_train = d; retrains += 1; a = accuracy(0)
        elif policy == 'drift' and a < theta:
            last_train = d; retrains += 1; a = accuracy(0)
        acc[d] = a
    return acc, retrains

sched, n_s = simulate_ct('scheduled', period=20)
drift, n_d = simulate_ct('drift', theta=0.80)
print(f'frozen    : mean acc {frozen.mean():.3f}, retrains 0')
print(f'scheduled : mean acc {sched.mean():.3f}, retrains {n_s}')
print(f'drift-trig: mean acc {drift.mean():.3f}, retrains {n_d}')

fig, ax = plt.subplots(figsize=(8.5, 4.3))
ax.plot(days, frozen*100, color=ROSE, lw=2, label='frozen (no CT)')
ax.plot(days, sched*100, color=BRAND, lw=2, label=f'scheduled ({n_s} retrains)')
ax.plot(days, drift*100, color=TEAL, lw=2, label=f'drift-triggered ({n_d} retrains)')
ax.axhline(80, color=YELLOW, ls='--', lw=1, alpha=0.6, label='threshold θ=80%')
ax.set_xlabel('day'); ax.set_ylabel('accuracy (%)'); ax.set_title('Continuous Training keeps a decaying model fresh')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

### Validate: drift-triggered CT matches scheduled with fewer retrains

Retraining only when accuracy dips below a threshold ("drift-triggered") should reach
comparable average accuracy to a fixed schedule, but *retrain fewer times* — spending
compute only when needed. We compare the two policies.

In [ ]:
acc_sched, n_sched = simulate_ct('scheduled', period=20)
acc_drift, n_drift = simulate_ct('drift', theta=0.80)
print(f'scheduled : mean acc {acc_sched.mean():.3f}, retrains {n_sched}')
print(f'drift-trig: mean acc {acc_drift.mean():.3f}, retrains {n_drift}')
assert acc_drift.mean() > frozen.mean(), 'any CT policy beats never retraining'
assert acc_sched.mean() > frozen.mean(), 'scheduled CT beats never retraining'
print('\n✅ continuous training arrests decay; drift-triggered spends retrains only when needed')

Drift-triggered CT often matches scheduled accuracy with fewer retrains — it retrains *when needed*, not on a calendar. But CT must be gated: a retrained model is only promoted if it beats the current one on a held-out set.

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **no retraining** | the model decays as data drifts (verified) |
| **retrain too often** | scheduled CT wastes compute; drift-triggered retrains only when needed |
| **no promotion gate** | automated pipelines can ship a regression — always gate on prod + floor |
| **train/serve skew** | the retraining pipeline must match the serving feature pipeline |
| **feedback-loop bias** | training on the deployed model's own outputs entrenches its errors |

Demo: the promotion gate only ships a candidate that beats production and clears the floor.

In [ ]:
# The promotion gate is the safety rail of automated retraining: a freshly retrained
# candidate ships ONLY if it beats production AND clears a quality floor. Without it, an
# automated pipeline could promote a regression. We test the gate's decision table.
def gate(cand, prod, floor):
    return cand > prod and cand >= floor
cases = [(0.85, 0.82, 0.80, True), (0.79, 0.82, 0.80, False), (0.81, 0.70, 0.85, False)]
for cand, prod, floor, expected in cases:
    got = gate(cand, prod, floor)
    print(f'candidate {cand}, prod {prod}, floor {floor} -> promote={got}')
    assert got == expected
print('\nThe gate promotes only a VALIDATED improvement -> automation never ships a regression.')

## ✏️ Your turn — the promotion gate

Implement `should_promote(candidate_acc, prod_acc, floor)` → promote only if the candidate beats production AND clears an absolute floor.

In [ ]:
def should_promote(candidate_acc, prod_acc, floor):
    """TODO(you): True iff candidate_acc > prod_acc and candidate_acc >= floor."""
    # TODO
    return ...


In [ ]:
assert should_promote(0.85, 0.82, 0.80) is True
assert should_promote(0.79, 0.82, 0.80) is False   # worse than prod
assert should_promote(0.81, 0.70, 0.85) is False   # below floor
print('✅ automated retraining only promotes a validated improvement.')

<details>
<summary>Solution</summary>

```python
def should_promote(candidate_acc, prod_acc, floor):
    return candidate_acc > prod_acc and candidate_acc >= floor
```

CT without this gate is a loop that can silently ship a worse model. Automate the *decision*, don't automate *around* it.
</details>

## Recap

- Models decay from **data drift** even when code is frozen — hence **Continuous Training**.
- **Scheduled** vs **drift-triggered** CT trade simplicity for efficiency; drift-triggered retrains when needed.
- Every retraining run must clear a **promotion gate** (beat prod + clear a floor) before deployment.
- Maturity levels 0→1→2 track automating the model, then the pipeline, then the pipeline's CI/CD.